
# 07 - Results, tables and figures

Collects every JSON file written by the other notebooks and regenerates **all** the tables
and figures in the paper. Nothing in the paper is transcribed by hand: rerun this notebook
and the paper's numbers are current.

It is designed to run on **partial** results, so it is useful while the grid is still
filling in - missing cells are shown as `--` rather than crashing.

In [ ]:

import os, json, glob, itertools
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT = "/content/drive/MyDrive/trpe"
except Exception:
    RESULT_ROOT = os.path.abspath("./trpe")

RESULTS, FIG_DIR = os.path.join(RESULT_ROOT, "results"), os.path.join(RESULT_ROOT, "figures")
TEX_DIR = os.path.join(RESULT_ROOT, "paper_tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TEX_DIR, exist_ok=True)

ARCHES = ["gcn", "gat"]
DATASETS = ["CSL", "CLUSTER", "CIFAR10"]
CONDS = [("M0", "no PE (control)"), ("M1", "LapPE"), ("M2", "RWPE"),
         ("M3", "T-RPE (ours)"), ("M4", "T-RPE + \\#sp (ours)")]

runs = []
for p in glob.glob(os.path.join(RESULTS, "*", "*", "*.json")):
    try:
        runs.append(json.load(open(p)))
    except Exception as e:
        print("skipped unreadable", p, e)
print(f"loaded {len(runs)} runs from {RESULTS}")


def cell(arch, dataset, cond):
    v = [100 * r["test_acc"] for r in runs
         if r.get("arch") == arch and r.get("dataset") == dataset and r.get("condition") == cond]
    return (np.mean(v), np.std(v), len(v)) if v else (None, None, 0)

In [ ]:
# ------------------------------- Table 1: main results -------------------------------
# CIFAR10-SP runs a single seed, so its std is 0 by construction rather than by
# measurement. A cell with n=1 is printed WITHOUT a +/- : reporting "46.62 +/- 0.00"
# would claim a zero-variance result we never measured.
DISPLAY = {"CSL": "CSL", "CLUSTER": "CLUSTER", "CIFAR10": "CIFAR10-SP"}

for arch in ARCHES:
    print(f"\n=== {arch.upper()} ===")
    print(f"{'condition':<24}" + "".join(f"{DISPLAY[d]:>18}" for d in DATASETS))
    print("-" * (24 + 18 * len(DATASETS)))
    for cond, label in CONDS:
        line = f"{cond + '  ' + label.replace(chr(92)+'#','#'):<24}"
        for d in DATASETS:
            m, s, n = cell(arch, d, cond)
            line += (f"{'--':>18}" if n == 0 else
                     f"{m:>12.2f} ± {s:<3.1f}" if n > 1 else f"{m:>13.2f} (n=1)")
        print(line)
    print(f"{'runs per cell':<24}" + "".join(f"{cell(arch, d, 'M0')[2]:>18}" for d in DATASETS))

def fmt(m, s, n, best):
    if n == 0: return "--"
    body = f"{m:.2f}\\,$\\pm$\\,{s:.2f}" if n > 1 else f"{m:.2f}"
    return f"\\textbf{{{body}}}" if best else body

tex = ["\\begin{tabular}{ll" + "c" * len(DATASETS) + "}", "\\toprule",
       "& & " + " & ".join(DISPLAY[d] for d in DATASETS) + " \\\\", "\\midrule"]
for arch in ARCHES:
    tex.append(f"\\multirow{{{len(CONDS)}}}{{*}}{{\\rotatebox{{90}}{{{arch.upper()}}}}}")
    bests = {d: max([cell(arch, d, c)[0] for c, _ in CONDS if cell(arch, d, c)[2]] or [None])
             for d in DATASETS}
    for cond, label in CONDS:
        cells = []
        for d in DATASETS:
            m, s, n = cell(arch, d, cond)
            cells.append(fmt(m, s, n, n and bests[d] is not None and abs(m - bests[d]) < 1e-9))
        tex.append(f"& {cond}~{label} & " + " & ".join(cells) + " \\\\")
    tex.append("\\midrule")
tex[-1] = "\\bottomrule"
tex.append("\\end{tabular}")
open(os.path.join(TEX_DIR, "table1_main.tex"), "w").write("\n".join(tex))
print("\nwritten ->", os.path.join(TEX_DIR, "table1_main.tex"))


In [ ]:
# ------------------------ Table 2: expressivity (from notebook 01) ------------------------
p = os.path.join(RESULTS, "expressivity", "expressivity.json")
if os.path.exists(p):
    E = json.load(open(p))
    print(f"{'witness pair':<42}{'1-WL':>11}{'SPD-WL':>11}{'SPcount-WL':>13}{'supports':>10}")
    print("-" * 87)
    for r in E["witness_pairs"]:
        print(f"{r['pair']:<42}{r['1-WL']:>11}{r['SPD-WL']:>11}{r['SPcount-WL']:>13}{r['prop']:>10}")
    w = E["prop4_witness"]
    print(f"\nProp. 4 minimal witness: {w['n']} nodes (exhaustive search found none smaller)")
    print(f"  G1 {w['G1']}\n  G2 {w['G2']}")

    # CSL ships 15 permuted copies per class, so a share of the colliding pairs are
    # ISOMORPHIC and no invariant can separate them. They sit in the denominator of the
    # raw percentage, so the rate among SEPARABLE pairs is reported alongside it.
    print("\nSeparation rate on the benchmarks:")
    for r in E["separation_rates"]:
        c, nei = r["wl_collisions"], r.get("neither_separated", 0)
        sep = c - nei
        f = lambda k: (f"{100*r[k]/c:.1f}%", f"{100*r[k]/sep:.1f}%") if c and sep else ("n/a", "n/a")
        sd, sc = f("spd_separated"), f("spc_separated")
        print(f"  {r['dataset']:<9} 1-WL-indistinguishable {c:>6} (isomorphic {nei:>5})  "
              f"SPD-WL {sd[0]:>6} ({sd[1]:>6} of separable)  "
              f"SPcount-WL {sc[0]:>6} ({sc[1]:>6})")

    tex = [r"\begin{tabular}{lccc}", r"\toprule",
           r"witness pair & 1-WL & \SPD{} & \SPC{} \\", r"\midrule"]
    mark = lambda s: r"\checkmark" if s == "separates" else r"--"
    for r in E["witness_pairs"]:
        nm = r["pair"].replace("_", r"\_").replace("x", r"$\times$").replace("#", r"\#")
        tex.append(f"{nm} & {mark(r['1-WL'])} & {mark(r['SPD-WL'])} & {mark(r['SPcount-WL'])} \\\\")
    tex += [r"\midrule",
            r"\multicolumn{4}{l}{\emph{1-WL-indistinguishable pairs separated, of those not isomorphic}} \\"]
    for r in E["separation_rates"]:
        c, nei = r["wl_collisions"], r.get("neither_separated", 0)
        sep = c - nei
        if not c:
            tex.append(f"{r['dataset']} (no 1-WL collisions) & -- & -- & -- \\\\")
        else:
            tex.append(f"{r['dataset']} ({sep} pairs) & -- & "
                       f"{100*r['spd_separated']/sep:.1f}\\% & {100*r['spc_separated']/sep:.1f}\\% \\\\")
    tex += [r"\bottomrule", r"\end{tabular}"]
    open(os.path.join(TEX_DIR, "table2_expressivity.tex"), "w").write("\n".join(tex))
    print("\nwritten ->", os.path.join(TEX_DIR, "table2_expressivity.tex"))
else:
    print("expressivity.json not found - run shared/01_expressivity.ipynb first")


In [ ]:
# --------------------------- Figure 2: ablations (notebook 05) ---------------------------
# NO M0 reference line. The ablation notebook trains 40 epochs on 1 seed while the main
# grid trains 60 on 3, so an ablation bar is not comparable to a main-grid M0 and drawing
# one across these bars invites exactly that comparison. Everything plotted here shares a
# budget, so the bars are comparable to EACH OTHER, which is what the attribution claim needs.
abl = [json.load(open(p)) for p in glob.glob(os.path.join(RESULTS, "ablations", "*.json"))]
if abl:
    by = {}
    for r in abl:
        by.setdefault(r["variant"], []).append(100 * r["test_acc"])

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    radii = [("r1", 1), ("r2", 2), ("r3", 3)]
    have = [(lbl, r) for lbl, r in radii if lbl in by]
    axes[0].plot([r for _, r in have], [np.mean(by[l]) for l, _ in have],
                 "o-", color="#3b6ea5", label="T-RPE")
    axes[0].set_xlabel("traversal radius $r$"); axes[0].set_ylabel("test accuracy (%)")
    axes[0].set_xticks([r for _, r in radii])
    axes[0].set_title("Reach of the traversal"); axes[0].legend(frameon=False, fontsize=8)

    order = ["rewire-only", "profile-only", "enc-raw", "enc-learnable", "weight-only",
             "enc-onehot", "layerwise", "r2"]
    show = [o for o in order if o in by]
    cols = ["#c76a4a" if o in ("rewire-only", "profile-only", "weight-only") else "#3b6ea5"
            for o in show]
    axes[1].barh(range(len(show)), [np.mean(by[o]) for o in show], color=cols)
    axes[1].set_yticks(range(len(show)))
    axes[1].set_yticklabels([("rewire-only (control)" if o == "rewire-only" else o) for o in show])
    lo = min(np.mean(by[o]) for o in show)
    axes[1].set_xlim(lo - 3, max(np.mean(by[o]) for o in show) + 1)
    axes[1].set_xlabel("test accuracy (%)"); axes[1].set_title("Which component matters")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "fig_ablations.pdf"), bbox_inches="tight")
    plt.show()
    print("saved ->", os.path.join(FIG_DIR, "fig_ablations.pdf"))

    if "rewire-only" in by and "r2" in by:
        gap = np.mean(by["r2"]) - np.mean(by["rewire-only"])
        print(f"\nATTRIBUTION: T-RPE(r=2) - rewire-only = {gap:+.2f} points (matched budget).")
        print("  A gap near zero would mean the benefit came from added connectivity rather")
        print("  than from the traversal signal it carries.")
    if "weight-only" in by and "profile-only" in by:
        print(f"CHANNELS: weight-only {np.mean(by['weight-only']):.2f} | "
              f"profile-only {np.mean(by['profile-only']):.2f} | both {np.mean(by['r2']):.2f}")
    print("\nNOTE: ablations use a shorter schedule (40 epochs, 1 seed) than the main grid")
    print("      (60 epochs, 3 seeds). Compare these bars only to each other.")


In [ ]:

# ------------------------ Figure: training curves per condition ------------------------
fig, axes = plt.subplots(len(ARCHES), len(DATASETS), figsize=(13, 6), squeeze=False)
colors = {"M0": "#888888", "M1": "#c76a4a", "M2": "#d9a441", "M3": "#3b6ea5", "M4": "#2f4858"}
for i, arch in enumerate(ARCHES):
    for j, d in enumerate(DATASETS):
        ax = axes[i][j]
        for cond, _ in CONDS:
            hs = [r["history"] for r in runs if r.get("arch") == arch
                  and r.get("dataset") == d and r.get("condition") == cond and r.get("history")]
            if not hs: continue
            L = min(len(h) for h in hs)
            curve = np.mean([[e["val_acc"] for e in h[:L]] for h in hs], axis=0) * 100
            ax.plot(range(1, L + 1), curve, label=cond, color=colors[cond], lw=1.4)
        ax.set_title(f"{arch.upper()} / {d}", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
        if j == 0: ax.set_ylabel("val accuracy (%)")
        if i == len(ARCHES) - 1: ax.set_xlabel("epoch")
        if i == 0 and j == 0: ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_curves.pdf"), bbox_inches="tight")
plt.show()
print("saved ->", os.path.join(FIG_DIR, "fig_curves.pdf"))

In [ ]:

# ------------------------------ completeness report ------------------------------
print("Grid status (runs completed per cell):\n")
print(f"{'':<8}" + "".join(f"{d:>12}" for d in DATASETS))
missing = 0
for arch in ARCHES:
    for cond, _ in CONDS:
        line = f"{arch}/{cond:<4}"
        for d in DATASETS:
            n = cell(arch, d, cond)[2]
            missing += (n == 0)
            line += f"{n:>12}"
        print(line)
print(f"\n{missing} of {len(ARCHES)*len(CONDS)*len(DATASETS)} cells still empty.")
print(f"\nAll paper assets regenerated:\n  tables  {TEX_DIR}\n  figures {FIG_DIR}")